# 03 - Modelagem Neural Multimodal: BenabderrahmaneNet

> **Iniciação Científica (IC) em Engenharia de Computação**  
> **Tema:** Detecção / Triagem de Transtorno do Espectro Autista (TEA) via Eye-Tracking e Deep Learning  
> **Objetivo Desta Etapa:** Implementar a arquitetura neural multimodal baseada em Benabderrahmane et al. (2024), com suporte a aceleração (GPU/CPU), pooling temporal adaptativo 1D para redução de custo computacional nas sequências longas e treinamento consistente de 5 épocas no Fold 1 (LOSO-CV).

---

##  Escopo e Otimizações
1. **Suporte a Dispositivo Acelerado:** Detecção dinâmica de hardware (`cuda` vs `cpu`).
2. **Otimização de Entrada Temporal (Downsampling 1D):** Camada `nn.AvgPool1d(kernel_size=5, stride=5)` no ramo de distâncias para reduzir o comprimento excessivo do padding de 10.370 para ~2.074 timesteps, acelerando drasticamente o processamento das GRUs sem perda de dinâmica.
3. **Ponderação de Classes:** Uso de `CrossEntropyLoss(weight=...)` balanceando a perda pelas frequências inversas das classes no conjunto de treino para mitigar o viés em direção à classe majoritária.
4. **Treinamento de 5 Épocas no Fold 1:** Treinamento com `batch_size=16` e `Adam(lr=3e-4)` em todos os sujeitos exceto o ID 1, seguido de avaliação e cálculo completo de métricas (Acurácia, F1-Macro, F1-Ponderado e Matriz de Confusão).


In [1]:
import sys
from pathlib import Path
import torch

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from baselines.benabderrahmane.model import BenabderrahmaneNet
from baselines.benabderrahmane.dataset import (
    BenabderrahmaneDataset,
    build_stimulus_mapping,
    filter_valid_groups,
    load_clean_parquet,
    precompute_dataset_cache,
    resolve_parquet_path,
)
from baselines.benabderrahmane.training import (
    compute_full_benchmark_metrics,
    get_device,
    run_loso_cv,
    train_single_fold,
    validate_forward_pass,
)
from baselines.benabderrahmane.visualization import plot_confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
get_device(device=device, verbose=True)
PARQUET_PATH = resolve_parquet_path()
print(f"Caminho da base Parquet: {PARQUET_PATH}")

PyTorch Version: 2.14.1+cu130
Dispositivo selecionado: cpu
  └─ CPU multiprocessada ativa (GPU não detectada neste kernel)
Caminho da base Parquet: /home/raphaell-arcanjo/Documents/ic-eye-tracking/data/processed/cilia_clean.parquet


In [2]:
df = load_clean_parquet(PARQUET_PATH, verbose=True)
stimulus_to_id = build_stimulus_mapping(df)
valid_groups, _, _ = filter_valid_groups(df, verbose=False)

dataset = BenabderrahmaneDataset(
    valid_keys=list(valid_groups.keys()),
    groups_dict=valid_groups,
    stim_map=stimulus_to_id,
)
print(f"Dataset instanciado com {len(dataset):,} sequências multimodais ({len(stimulus_to_id)} estímulos únicos).")

Base Parquet carregada em 0.18s | Shape: (1351157, 8)
Total de estímulos únicos: 148
Total de grupos válidos para o Dataset: 2,115 amostras
Dataset instanciado com 2,115 sequências multimodais.


## 2. Implementação da Arquitetura Multimodal Otimizada (`BenabderrahmaneNet`)

A arquitetura original de Benabderrahmane et al. (2024) foi enriquecida com uma camada de **Pooling Temporal 1D Adaptativo (`nn.AvgPool1d(kernel_size=5, stride=5)`)** no ramo de distâncias. Isso comprime o vetor de 10.370 para **2.074 timesteps**, preservando a integral de movimento ocular e reduzindo em ~5x o tempo de computação por batch.

```
                  ┌──────────────────────────────────────────────┐
                  │                 ENTRADAS                     │
                  └──────┬──────────────┬──────────────┬─────────┘
                         │              │              │
       [Scan-path 2D]    │ [Distâncias] │  [Durações]  │ [Escalares]
       (B, 1, 400, 600)  │ (B, 10370)   │  (B, 1313)   │   (B, 2)
              │          │      │       │      │       │     │
              │          │ AvgPool1d(5) │      │       │     │
              │          │ (B, 2074, 1) │      │       │     │
       ┌──────▼─────┐    │ ┌────▼─────┐ │ ┌────▼─────┐ │ ┌───▼────┐
       │  Ramo CNN  │    │ │ Ramo GRU1│ │ │ Ramo GRU2│ │ │Ramo MLP│
       │(Conv+BN+MP)│    │ │ (Hidden) │ │ │ (Hidden) │ │ (Linear) │
       └──────┬─────┘    │ └────┬─────┘ │ └────┬─────┘ │ └───┬────┘
              │          │      │       │      │       │     │
         Vetor (128)     │  Vetor (64)  │  Vetor (64)  │ Vetor(16)
              │          │      │       │      │       │     │
              └──────────┼──────┴───────┼──────┴───────┼─────┘
                         ▼              ▼              ▼
                  ┌──────────────────────────────────────────────┐
                  │          Fusão Tardia (Concatenação)         │
                  │             Vetor Latente: 272-D             │
                  └──────────────────────┬───────────────────────┘
                                         ▼
                  ┌──────────────────────────────────────────────┐
                  │      Classificador MLP (Linear+ReLU+Dropout) │
                  └──────────────────────┬───────────────────────┘
                                         ▼
                               Saída: Logits (B, 2)
                                 [Classe 0 / 1]
```


## 3. Validação do Forward Pass (Teste de Sanidade no Dispositivo)

In [3]:
out_sanity = validate_forward_pass(dataset, device=device, batch_size=2, seed=42)

Dispositivo de execução : cuda
Shape de saída          : (2, 2)
Logits gerados          :
[[-0.19224621 -0.03720935]
 [-0.11850895 -0.00155723]]
Tempo do forward pass   : 1400.42 ms
Total de parâmetros     : 275,682 treináveis
✓ Teste de sanidade aprovado com sucesso no dispositivo configurado!


## 4. Treinamento de 5 Épocas no Fold 1 (Teste: Participante ID 1)

- **Particionamento:** Treino com todos os participantes exceto ID 1 ($N_{treino} = 2.085$) e Teste no Participante ID 1 ($N_{teste} = 30$, classe ASD).
- **Hiperparâmetros:** `batch_size=16`, `lr=3e-4`, `num_epochs=5` e ponderação inversa de classes na `CrossEntropyLoss`.

In [4]:
BATCH_SIZE, LR, EPOCHS = 16, 3e-4, 5
fold1_result = train_single_fold(
    valid_groups, stimulus_to_id, test_pid=1, num_epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=device
)

CONFIGURAÇÃO DO FOLD 1 (TESTE: PARTICIPANTE ID 1)
• Amostras de Treino : 2,085 | TD: 1269 (60.9%), ASD: 816 (39.1%)
• Pesos de Classe    : TD=0.8215, ASD=1.2776
• Amostras de Teste  : 30 (Participante 1)
• Dispositivo        : cuda

Iniciando treinamento de 5 épocas (131 batches/época, batch_size=16)...

• Época [1/5] | Loss Médio de Treino: 0.6791 | Tempo: 26.6s
• Época [2/5] | Loss Médio de Treino: 0.6344 | Tempo: 25.3s
• Época [3/5] | Loss Médio de Treino: 0.6160 | Tempo: 25.9s
• Época [4/5] | Loss Médio de Treino: 0.6066 | Tempo: 25.9s
• Época [5/5] | Loss Médio de Treino: 0.5917 | Tempo: 26.2s

AVALIAÇÃO FINAL NA 5ª ÉPOCA - PARTICIPANTE ID 1
• Loss de Teste : 0.4873
• Tempo de Teste: 0.23s
---------------------------------------------------------------------------
MÉTRICAS DE AVALIAÇÃO DE DESEMPENHO (SKLEARN)
• Acurácia (Accuracy)         : 73.33%
• F1-Score (Macro)            : 0.4231
• F1-Score (Ponderado)        : 0.8462
---------------------------------------------------------

## 5. Execução de Sanidade do LOSO-CV (2 Primeiros Folds)

Validação prática do protocolo *Leave-One-Subject-Out* (`run_loso_cv`) nos 2 primeiros sujeitos (IDs 1 e 2) por 2 épocas.

In [5]:
BATCH_SIZE, LR, SANITY_EPOCHS, MAX_FOLDS = 16, 3e-4, 2, 2
loso_summary = run_loso_cv(
    dataset=dataset, num_epochs=SANITY_EPOCHS, batch_size=BATCH_SIZE, lr=LR, max_folds=MAX_FOLDS, device=device
)

Iniciando LOSO-CV limitado aos primeiros 2 folds de teste (total: 57)...

FOLD [1/2] - SUJEITO DE TESTE ID = 1
• Treino: 2,085 amostras (TD: 1269, ASD: 816) | Pesos: [TD=0.822, ASD=1.278]
• Teste : 30 amostras (Participante 1 | Classe: ASD)
  └─ Época [1/2] | Loss Treino: 0.6705 | Tempo: 26.3s
  └─ Época [2/2] | Loss Treino: 0.6292 | Tempo: 26.0s
  └─ Resumo Fold 1: Loss Teste: 0.5976 | Acurácia: 60.0% (18/30) | Duração: 52.6s

FOLD [2/2] - SUJEITO DE TESTE ID = 2
• Treino: 2,093 amostras (TD: 1269, ASD: 824) | Pesos: [TD=0.825, ASD=1.270]
• Teste : 22 amostras (Participante 2 | Classe: ASD)
  └─ Época [1/2] | Loss Treino: 0.6646 | Tempo: 26.6s
  └─ Época [2/2] | Loss Treino: 0.6349 | Tempo: 26.2s
  └─ Resumo Fold 2: Loss Teste: 0.5227 | Acurácia: 59.1% (13/22) | Duração: 53.0s

LOSO-CV concluído em 106.2s (1.77 min).

RELATÓRIO CONSOLIDADO DE MÉTRICAS (BENCHMARK BENABDERRAHMANE ET AL.)
• Total de Amostras Avaliadas : 52
• Acurácia Global (Accuracy) : 59.62%
• Sensibilidade / Recall (A

## 6. Pré-computação dos Tensores em Memória RAM (Aceleração para GPU)

Pré-computa as 2.115 amostras multimodais na memória RAM para eliminar gargalos de CPU durante os 57 folds do LOSO-CV.

In [6]:
precomputed_cache = precompute_dataset_cache(dataset, verbose=True)

Pré-computando tensores para 2,115 amostras...
✓ Pré-computação concluída em 15.7s! Tensores prontos na memória RAM.


## 7. Execução do Benchmark Completo dos 57 Participantes (LOSO-CV)

Executa o protocolo *subject-independent* completo para todos os 57 participantes válidos do dataset Cilia com 5 épocas por fold, reportando as métricas consolidadas (`compute_full_benchmark_metrics`) e a matriz de confusão (`plot_confusion_matrix`).

In [7]:
BATCH_SIZE, LR, EPOCHS = 16, 3e-4, 5
loso_full_results = run_loso_cv(
    dataset=dataset, precomputed_cache=precomputed_cache, num_epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=device
)
plot_confusion_matrix(loso_full_results["confusion_matrix"])

Iniciando LOSO-CV COMPLETO com todos os 57 participantes (57 folds)...

FOLD [1/57] - SUJEITO DE TESTE ID = 1
• Treino: 2,085 amostras (TD: 1269, ASD: 816) | Pesos: [TD=0.822, ASD=1.278]
• Teste : 30 amostras (Participante 1 | Classe: ASD)
  └─ Época [1/5] | Loss Treino: 0.6709 | Tempo: 11.8s
  └─ Época [2/5] | Loss Treino: 0.6295 | Tempo: 12.1s
  └─ Época [3/5] | Loss Treino: 0.6083 | Tempo: 12.1s
  └─ Época [4/5] | Loss Treino: 0.5925 | Tempo: 11.7s
  └─ Época [5/5] | Loss Treino: 0.5858 | Tempo: 11.5s
  └─ Resumo Fold 1: Loss Teste: 0.7793 | Acurácia: 50.0% (15/30) | Duração: 59.2s

FOLD [2/57] - SUJEITO DE TESTE ID = 2
• Treino: 2,093 amostras (TD: 1269, ASD: 824) | Pesos: [TD=0.825, ASD=1.270]
• Teste : 22 amostras (Participante 2 | Classe: ASD)
  └─ Época [1/5] | Loss Treino: 0.6632 | Tempo: 11.5s
  └─ Época [2/5] | Loss Treino: 0.6318 | Tempo: 11.5s
  └─ Época [3/5] | Loss Treino: 0.6027 | Tempo: 11.7s
  └─ Época [4/5] | Loss Treino: 0.5983 | Tempo: 11.8s
  └─ Época [5/5] | Loss